# deterministic_slm Kaggle/Colab reproduction

This notebook captures the repo's existing `make hello` and `make demo` output, then runs a separate Hugging Face Transformers repeatability check for notebook GPU runtimes.

`make demo` always includes the constructed toy divergence. Its live Ollama/OpenAI-compatible probe is optional evidence: if no server is available at `http://localhost:11434/v1`, the demo reports that no live backend result was collected. The Transformers section below is a notebook-native comparison path, not the same backend as `make demo`.

## 1. Locate or clone the repo

Run this notebook from the repository root when possible. If Kaggle or Colab starts in an empty workspace, this cell clones the public repo into the notebook workspace and changes into it.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/SteffenHaeussler/deterministic_slm.git"
REPO_DIR = Path("/kaggle/working/deterministic_slm") if Path("/kaggle").exists() else Path("/content/deterministic_slm")

def has_repo_files(path: Path) -> bool:
    return (path / "Makefile").exists() and (path / "pyproject.toml").exists()

cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd.parent.parent, REPO_DIR]
repo_root = next((path for path in candidates if has_repo_files(path)), None)

if repo_root is None:
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    if REPO_DIR.exists() and not has_repo_files(REPO_DIR):
        raise RuntimeError(f"{REPO_DIR} exists but does not look like this repo; remove it or set REPO_DIR to another path.")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    repo_root = REPO_DIR

os.chdir(repo_root)
print(f"repo_root={Path.cwd()}")
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=False)

## 2. Runtime diagnostics

These diagnostics make the notebook output useful when shared. They show whether the runtime has an NVIDIA GPU and whether PyTorch can see CUDA.

In [ ]:
import platform
import shutil
import sys

print(f"python={sys.version}")
print(f"platform={platform.platform()}")
print(f"cwd={Path.cwd()}")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found; this runtime may not have an NVIDIA GPU attached.")

try:
    import torch

    print(f"torch={torch.__version__}")
    print(f"torch.cuda.is_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"cuda_device={torch.cuda.get_device_name(0)}")
except Exception as error:
    print(f"torch diagnostics unavailable before dependency install: {error}")

## 3. Install project dependencies

The repo uses `uv`. This installs `uv` if needed, then runs the same sync step used locally.

In [ ]:
if shutil.which("uv") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

uv = shutil.which("uv")
if uv is None:
    raise RuntimeError("uv installed, but the uv executable is not on PATH. Restart the runtime and rerun this cell.")

subprocess.run([uv, "--version"], check=True)
subprocess.run([uv, "sync"], check=True)

## 4. Capture `make hello`

`make hello` is the reliable local reproduction. It shows the constructed float32 reduction grouping divergence and should exit successfully.

In [ ]:
def run_and_capture(command: list[str]) -> subprocess.CompletedProcess[str]:
    print(f"$ {' '.join(command)}")
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    print("--- stdout ---")
    print(result.stdout.rstrip() or "<empty>")
    print("--- stderr ---")
    print(result.stderr.rstrip() or "<empty>")
    print(f"--- exit_code: {result.returncode} ---")
    return result

hello_result = run_and_capture(["make", "hello"])
assert hello_result.returncode == 0, "make hello should succeed"

## 5. Capture `make demo`

`make demo` first prints the same constructed divergence, then calls the live Ollama/OpenAI-compatible probe. In a Transformers-only Kaggle/Colab notebook, the live probe usually reports `live_status: unavailable` because no Ollama server is running. That means no live backend evidence was collected; it is not evidence of nondeterminism.

In [ ]:
demo_result = run_and_capture(["make", "demo"])
assert demo_result.returncode == 0, "make demo should succeed even when the optional live backend is unavailable"

## 6. Install Transformers dependencies

This section is separate from `make demo`. It loads a small Hugging Face model directly with Transformers so the notebook can exercise the available GPU without standing up an OpenAI-compatible server.

In [ ]:
packages = ["transformers>=4.37.0", "accelerate>=0.26.0"]
try:
    import torch  # noqa: F401
except Exception:
    packages.append("torch")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

## 7. Transformers repeatability sanity check

The default model is `Qwen/Qwen2.5-0.5B-Instruct`. The generation settings use greedy decoding (`do_sample=False`) and a fixed seed. The output hashes make it easy to compare repeated in-process runs. Identical hashes here mean no divergence was observed in this narrow run; they do not prove backend-wide determinism across fresh processes, hardware, batching, or serving stacks.

In [ ]:
import hashlib
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
PROMPT = "hi, how are you?"
REPEAT = 5
MAX_NEW_TOKENS = 32
SEED = 0

set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()

messages = [{"role": "user", "content": PROMPT}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

records = []
for index in range(1, REPEAT + 1):
    set_seed(SEED)
    started = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency_seconds = time.perf_counter() - started
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    text_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
    records.append({"text": text, "hash": text_hash, "latency_seconds": latency_seconds})

for index, record in enumerate(records, start=1):
    print(f"run: {index}")
    print(f"text: {record['text']}")
    print(f"output_hash: {record['hash']}")
    print(f"latency_seconds: {record['latency_seconds']:.6f}")
    print()

unique_hashes = sorted({record["hash"] for record in records})
print(f"transformers_backend: {MODEL_ID}")
print(f"device: {model.device}")
print(f"repeat: {REPEAT}")
print(f"unique_output_count: {len(unique_hashes)}")
print(f"output_hashes: {unique_hashes}")
if len(unique_hashes) == 1:
    print("interpretation: no divergence observed in this in-process greedy run.")
    print("interpretation_scope: this does not prove backend-wide determinism across processes, hardware, batching, or serving stacks.")
else:
    print("interpretation: divergence observed in this in-process greedy run.")